In [1]:
from model_arch import *

In [2]:
from bpetokenizer import *
tokenizer = load_bpe_tokenizer()

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SEP = tokenizer.encode("####")[0] 
END = tokenizer.encode("</s>")[0]

In [5]:
test = pickle.load(open("test_data_processed.pkl", "rb"))

In [ ]:
model = DecoderLM(LMConfig(vocab_size=50260)).to(device)
model.load_state_dict(torch.load("checkpoint.pt", map_location=device))
model.eval()

In [ ]:
prompts = []
head = 722 // 2
for e in test:
    doc = e["tokens"]
    if len(doc) > 722:
        doc = doc[:head] + doc[-head:] 
    prompts.append(doc + [SEP])

groups = dict()
for i, p in enumerate(prompts):
    if len(p) in groups:
        groups[len(p)].append(i)
    else:
        groups[len(p)] = [i]

In [ ]:
import json
import torch

@torch.no_grad()
def generate(prompt_list, max_new=300):
    x = torch.tensor(prompt_list, device=device)
    B, L = x.shape
    
    out = torch.zeros(B, max_new, dtype=torch.long, device=device)
    finished = torch.zeros(B, dtype=torch.bool, device=device)
    
    with torch.autocast("cuda", dtype=torch.float16):
        for n in range(max_new):
            logits = model(x)
            logits = logits[:, -1, :].float()
            
            recent = x[:, -64:]
            vals = logits.gather(1, recent)
            logits.scatter_(1, recent, torch.where(vals > 0, vals / 1.1, vals * 1.1))
                
            nxt = logits.argmax(dim=-1)
            out[:, n] = nxt
            finished |= (nxt == END)
            
            if finished.all(): 
                break
                
            x = torch.cat([x, nxt.unsqueeze(1)], dim=1)
            
    return [r[:r.index(END)] if END in r else r for r in out[:, :n + 1].tolist()]

In [ ]:
summaries = {}
total = len(prompts)
BS = 16

for g, (L, idxs) in enumerate(groups.items()):
    for s in range(0, len(idxs), BS):
        batch = idxs[s:s + BS]
        batch_prompts = [prompts[i] for i in batch]
        
        for i, out_tokens in zip(batch, generate(batch_prompts)):
            summaries[i] = tokenizer.decode(out_tokens).strip()

        done = len(summaries)

In [ ]:
out_json = []
for i, e in enumerate(test):
    text = summaries.get(i, "")
    if not text:
        text = tokenizer.decode(e["tokens"][:300])
    out_json.append({"id": e["id"], "summary": text})

with open("prediction.json", "w") as f:
    json.dump(out_json, f)